# AI Systems Engineering Workshop — Start Here


<a href="https://colab.research.google.com/" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **AI Systems Engineering Workshop** &middot; Runs on Google Colab &middot; Requires a Google Gemini API key.


Nine Colab notebooks that build a production-grade AI system layer by layer, using a single
running example: **SupportCo**, a fictional company with policy documents, a ticket database
and customers who ask hard questions.

Every notebook is self-contained: it installs its own dependencies, rebuilds the corpus and the
database, and runs top to bottom in a fresh Colab runtime.

## The stack

```text
AI SYSTEMS ENGINEERING STACK
  +---------------------------------------------------+
  | 6. Responsible AI & Governance   (rules for all)  |
  +---------------------------------------------------+
  | 5. Loop Engineering   (budgets, approval, idem.)  |
  +---------------------------------------------------+
  | 4. Agentic Systems    (state, tools, ODA loop)    |
  +---------------------------------------------------+
  | 3. Advanced RAG       (hybrid, rerank, corrective)|
  +---------------------------------------------------+
  | 2. Retrieval          (chunks, embeddings, top-k) |
  +---------------------------------------------------+
  | 1. LLM Foundation     (tokens, prompts, JSON)     |
  +---------------------------------------------------+
   cross-cutting: 7. Evaluation  8. Frameworks  9. AI-assisted dev
```

## Notebooks

| # | Notebook | Time | What you build |
|---|---|---|---|
| 1 | `01_LLM_Foundation.ipynb` | 90 min | Tokens, context, temperature, prompts, schema-validated JSON, RAG vs fine-tuning |
| 2 | `02_Retrieval.ipynb` | 120 min | Chunking, embeddings, cosine search, evidence selection, citations, metadata filters |
| 3 | `03_Advanced_RAG.ipynb` | 150 min | BM25 + hybrid, cross-encoder rerank, decomposition, multi-query, corrective loop |
| 4 | `04_Agentic_Systems.ipynb` | 150 min | Agent state, tool registry, validation gates, Observe–Decide–Act loop, ownership checks |
| 5 | `05_Loop_Engineering.ipynb` | 180 min | Budgets, failure taxonomy, SHA-256 approval binding, idempotency, checkpoints, traces |
| 6 | `06_Responsible_AI_and_Governance.ipynb` | 150 min | Grounding, injection defence, privacy, audit, risk register, system card |
| 7 | `07_Evaluation.ipynb` | 150 min | Golden sets, retrieval/generation metrics, security tests, release gates |
| 8 | `08_Frameworks_and_Interoperability.ipynb` | 150 min | LangGraph, MCP server + client, A2A task lifecycle, domain contracts |
| 9 | `09_AI_Assisted_Software_Development.ipynb` | 120 min | Problem definition, acceptance criteria, plan → review → test → regression → disclose |

**Run them in order.** Each one assumes the vocabulary and the controls introduced by the
previous ones.

## Before you start

1. Get a **Google Gemini API key** from [Google AI Studio](https://aistudio.google.com/app/apikey).
   The free tier is enough for the whole workshop.
2. Open a notebook in Colab.
3. Click the **key icon** in the left sidebar → *Add new secret* → name it `GOOGLE_API_KEY`,
   paste your key, enable *Notebook access*.
4. Run the cells from the top. If you skip the setup cell, everything below it will fail.

If you have not added the secret, the setup cell falls back to a password prompt.

> **Never hard-code an API key into a notebook you share.** Notebooks are copied, forked and
> screenshotted far more than source files are.

In [ ]:
# Quick check that your key works. Run this once before the workshop starts.
%pip install -q "google-genai>=1.0.0"

import os, getpass
from google import genai

key = os.environ.get("GOOGLE_API_KEY")
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
    except Exception:
        key = None
if not key:
    key = getpass.getpass("Paste your GOOGLE_API_KEY: ")

client = genai.Client(api_key=key)
print("chat  :", client.models.generate_content(
    model="gemini-3.6-flash", contents="Say OK").text.strip())
print("embed :", len(client.models.embed_content(
    model="gemini-embedding-2", contents="hello").embeddings[0].values), "dimensions")
print("\nYou are ready. Open 01_LLM_Foundation.ipynb.")

## The running example

Every notebook uses the same synthetic company so the concepts compound instead of restarting.

```text
  SupportCo
    8 policy documents          refunds, shipping, billing, accounts, a how-to guide
      - one SUPERSEDED version  (2023 refund policy)  -> version conflicts, outdated evidence
      - one INTERNAL document   (escalation matrix)   -> permission filtering
      - one HOSTILE document    (community note)      -> prompt-injection exercises
    4 customers                 u_alice, u_bob, u_carol
    4 tickets in SQLite         ownership, status, idempotency, checkpoints, traces
```

All data is invented for this workshop. There are no real people, companies or tickets in it.

## Teaching notes

- **Every notebook opens with a mental map.** Show it, walk the boxes, and only then run code.
  Learners who cannot place a concept in the stack will not retain it.
- **Run the failures.** The demonstrations of hallucination, injection, double-charging and
  budget exhaustion are the lesson. Do not skip them to save time.
- **The recurring sentence** is *"the model proposes, the runtime decides."* Repeat it until it
  becomes boring; it is the thesis of the whole workshop.
- **Lab exercises** are at the end of every notebook. Pick two or three per session rather than
  attempting all of them.
- **API cost**: the whole workshop runs comfortably inside the Gemini free tier. Notebooks 3
  and 7 make the most calls; if you are rate-limited, reduce the golden-set size.
- **Offline fallback**: there is none by design. Every notebook calls a real model, because the
  failure modes being taught only appear with a real model.

## Files a notebook may create in your runtime

| File | Created by | Purpose |
|---|---|---|
| `supportco.db` | Notebooks 4, 5, 6, 7, 8 | tickets, idempotency receipts, checkpoints, traces |
| `mcp_server.py` | Notebook 8 | the MCP server launched over stdio |
| `refunds_project/` | Notebook 9 | the code, tests and snapshots for the AI-assisted dev lab |

All of them live in the Colab runtime and disappear when it is recycled. Nothing is written
outside the runtime.